In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Artificial Intelligence Rasikh Ali - Lab 11: Classification and Model Evaluation\n",
    "\n",
    "This lab demonstrates the application and evaluation of multiple classification algorithms on the project data. Since the original target (`Price`) is continuous (a regression problem), we must first convert it into a categorical variable (`Price_Class`) to enable classification."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.naive_bayes import BernoulliNB, GaussianNB, MultinomialNB\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from sklearn.tree import DecisionTreeClassifier\n",
    "from sklearn.neighbors import KNeighborsClassifier\n",
    "from sklearn import metrics\n",
    "from sklearn.metrics import accuracy_score\n",
    "\n",
    "# Load the dataset\n",
    "data = pd.read_csv('processed_data.csv')\n",
    "print(\"Data loaded.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Data Transformation: Converting Regression to Classification\n",
    "The original target variable `Price` is continuous. To apply the requested classification models, we create `Price_Class` by binning the price values."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Determine price quantiles for classification bins\n",
    "bins = data['Price'].quantile([0.25, 0.50, 0.75]).tolist()\n",
    "bins = [data['Price'].min()] + bins + [data['Price'].max()]\n",
    "\n",
    "# Define labels for the four classes\n",
    "labels = [0, 1, 2, 3] # Low, Medium-Low, Medium-High, High\n",
    "\n",
    "# Create the new target column\n",
    "data['Price_Class'] = pd.cut(data['Price'], bins=bins, labels=labels, include_lowest=True)\n",
    "\n",
    "print(\"Original Price distribution:\")\n",
    "print(data['Price'].describe())\n",
    "print(\"\\nNew Price Class distribution:\")\n",
    "print(data['Price_Class'].value_counts())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Splitting into X and y (Features and Target)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Features (all columns except the original 'Price' and the new 'Price_Class')\n",
    "x = data.drop(['Price', 'Price_Class'], axis=1)\n",
    "x.shape\n",
    "\n",
    "# Target (the new 'Price_Class')\n",
    "y = data['Price_Class'].astype(int) # Ensure target is integer type\n",
    "y.shape\n",
    "\n",
    "print(f\"X (Features) shape: {x.shape}\")\n",
    "print(f\"y (Target) shape: {y.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Converting Object columns into Int columns\n",
    "Your data is already pre-processed. This step confirms no object columns need conversion."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "cat_columns = x.select_dtypes(['object']).columns\n",
    "\n",
    "if len(cat_columns) > 0:\n",
    "    # Apply factorization if object columns existed\n",
    "    x[cat_columns] = x[cat_columns].apply(lambda col: pd.factorize(col)[0])\n",
    "    print(f\"Converted object columns: {list(cat_columns)}\")\n",
    "else:\n",
    "    print(\"No object columns found. Features are already numerical.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Splitting Data into Train and Test\n",
    "Splitting the data with a 70% train and 30% test ratio, without shuffling."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size=0.3, shuffle=False)\n",
    "\n",
    "print(f\"Train set size (X_train): {X_train.shape}\")\n",
    "print(f\"Test set size (X_test): {X_test.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Applying Different Classifiers and Evaluating Scores"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "scores = {}\n",
    "classifiers = {\n",
    "    'Bernoulli': BernoulliNB(),\n",
    "    'Random Forest': RandomForestClassifier(random_state=42),\n",
    "    'Gaussian': GaussianNB(),\n",
    "    'Decision Tree': DecisionTreeClassifier(random_state=42),\n",
    "    'Multinomial': MultinomialNB(),\n",
    "    'KNeighbors': KNeighborsClassifier(n_neighbors=5)\n",
    "}\n",
    "\n",
    "for name, classifier in classifiers.items():\n",
    "    print(f\"\\n--- {name} Classifier ---\")\n",
    "    \n",
    "    # Train the model\n",
    "    classifier.fit(X_train, Y_train)\n",
    "    \n",
    "    # Predict on the test set\n",
    "    Y_pred = classifier.predict(X_test)\n",
    "    \n",
    "    # Calculate metrics\n",
    "    accuracy = accuracy_score(Y_test, Y_pred)\n",
    "    precision = metrics.precision_score(Y_test, Y_pred, average='weighted', zero_division=0)\n",
    "    recall = metrics.recall_score(Y_test, Y_pred, average='weighted', zero_division=0)\n",
    "    f1 = metrics.f1_score(Y_test, Y_pred, average='weighted', zero_division=0)\n",
    "    \n",
    "    # Store results\n",
    "    scores[name] = {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}\n",
    "    \n",
    "    # Print results (matching the required format)\n",
    "    print('Accuracy: %f' % accuracy)\n",
    "    print('Precision: %f' % precision)\n",
    "    print('Recall: %f' % recall)\n",
    "    print('F1 score: %f' % f1)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Plotting Line Graph Showcasing all the Scores of applied Classifiers"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "x_labels = list(scores.keys())\n",
    "b_accuracy = scores['Bernoulli']['accuracy']\n",
    "r_accuracy = scores['Random Forest']['accuracy']\n",
    "g_accuracy = scores['Gaussian']['accuracy']\n",
    "d_accuracy = scores['Decision Tree']['accuracy']\n",
    "m_accuracy = scores['Multinomial']['accuracy']\n",
    "k_accuracy = scores['KNeighbors']['accuracy']\n",
    "\n",
    "b_precision = scores['Bernoulli']['precision']\n",
    "r_precision = scores['Random Forest']['precision']\n",
    "g_precision = scores['Gaussian']['precision']\n",
    "d_precision = scores['Decision Tree']['precision']\n",
    "m_precision = scores['Multinomial']['precision']\n",
    "k_precision = scores['KNeighbors']['precision']\n",
    "\n",
    "b_recall = scores['Bernoulli']['recall']\n",
    "r_recall = scores['Random Forest']['recall']\n",
    "g_recall = scores['Gaussian']['recall']\n",
    "d_recall = scores['Decision Tree']['recall']\n",
    "m_recall = scores['Multinomial']['recall']\n",
    "k_recall = scores['KNeighbors']['recall']\n",
    "\n",
    "b_f1 = scores['Bernoulli']['f1']\n",
    "r_f1 = scores['Random Forest']['f1']\n",
    "g_f1 = scores['Gaussian']['f1']\n",
    "d_f1 = scores['Decision Tree']['f1']\n",
    "m_f1 = scores['Multinomial']['f1']\n",
    "k_f1 = scores['KNeighbors']['f1']\n",
    "\n",
    "# Create arrays for plotting\n",
    "accuracy_scores = [b_accuracy, r_accuracy, g_accuracy, d_accuracy, m_accuracy, k_accuracy]\n",
    "precision_scores = [b_precision, r_precision, g_precision, d_precision, m_precision, k_precision]\n",
    "recall_scores = [b_recall, r_recall, g_recall, d_recall, m_recall, k_recall]\n",
    "f1_scores = [b_f1, r_f1, g_f1, d_f1, m_f1, k_f1]\n",
    "\n",
    "plt.figure(figsize=(16, 10))\n",
    "\n",
    "plt.plot(x_labels, accuracy_scores, marker='o', label='Accuracy')\n",
    "plt.plot(x_labels, precision_scores, marker='o', label='Precision')\n",
    "plt.plot(x_labels, recall_scores, marker='o', label='Recall')\n",
    "plt.plot(x_labels, f1_scores, marker='o', label='F1')\n",
    "\n",
    "plt.title(\"Scores of Applied Classifiers\")\n",
    "plt.xlabel(\"Classifiers\")\n",
    "plt.ylabel(\"Score Value\")\n",
    "plt.grid(True)\n",
    "plt.legend()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Plotting Bar Graph Showcasing F1 Scores of applied Classifiers"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Get F1 scores and labels from the collected data\n",
    "tick_label = list(scores.keys())\n",
    "height = [s['f1'] for s in scores.values()]\n",
    "left = np.arange(len(tick_label))\n",
    "\n",
    "plt.figure(figsize=(16, 10))\n",
    "\n",
    "# plotting a bar chart\n",
    "plt.bar(left, height, tick_label=tick_label, width=0.9, color=['#08737f', '#00898a', '#089f8f', '#39b48e', '#64c987', '#92dc7e'])\n",
    "\n",
    "# naming the x-axis\n",
    "plt.xlabel('Classifiers')\n",
    "# naming the y-axis\n",
    "plt.ylabel('F1 Scores')\n",
    "# plot title\n",
    "plt.title('F1 Scores of Applied Classifiers')\n",
    "\n",
    "# function to show the plot\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.x"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}